# Stretch: more joins, worked solutions

The answers to the stretch notebook. Try each one before looking.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Stretch 1: who did I not play in 2025?

The obvious version first. It runs, it looks fine, and it is wrong.

In [ ]:
run("""
SELECT a.artist_name, COUNT(p.play_id) AS plays_2025
FROM artists a
LEFT JOIN plays p ON p.artist_id = a.artist_id
WHERE p.played_at >= '2025-01-01'
GROUP BY a.artist_id
ORDER BY plays_2025, a.artist_name
""", limit=5)

**35 rows, and nobody has 0 plays.** We asked for 40 artists.

Why: `WHERE` runs *after* the join. For an artist with no 2025 plays, the
left join produces rows whose `played_at` is either a 2024 date or `NULL`.
The `WHERE` throws all of those away, so the artist vanishes. The left join
did its job and the `WHERE` undid it. A filter on the right-hand table in
the `WHERE` quietly turns a left join back into an inner join.

The fix: move the condition into the `ON`. Now it is part of the matching
rule ("match this artist's plays, *and only the 2025 ones*"), and an artist
with no match is still kept, with `NULL`s.

In [ ]:
run("""
SELECT a.artist_name, COUNT(p.play_id) AS plays_2025
FROM artists a
LEFT JOIN plays p
       ON p.artist_id = a.artist_id
      AND p.played_at >= '2025-01-01'
GROUP BY a.artist_id
ORDER BY plays_2025, a.artist_name
""", limit=7)

All 40 artists are back, and five of them have 0 plays in 2025.

Now only the ones with no match:

In [ ]:
run("""
SELECT a.artist_name, a.genre
FROM artists a
LEFT JOIN plays p
       ON p.artist_id = a.artist_id
      AND p.played_at >= '2025-01-01'
WHERE p.play_id IS NULL
ORDER BY a.artist_name
""")

The same five. This shape, a left join followed by `WHERE right.key IS
NULL`, is the standard way to find "things with no match": customers with
no orders, users who never logged in, and so on.

**Why is this `WHERE` fine, when the first one was not?** Both run after
the join. The first one kept rows *with* a 2025 date, which threw away
exactly the unmatched rows. This one keeps *only* the unmatched rows,
which is what we wanted. A `WHERE` on the right-hand table is not always
wrong; you just have to know which rows it is keeping.

Two of the five, Iron Fernway and Gravel Choir, have never been played at
all. The other three were played in 2024 and then dropped. Stretch 3
separates them.

## Stretch 2: artists from the same country

In [ ]:
run("""
SELECT a1.country,
       a1.artist_name AS artist,
       a2.artist_name AS same_country_as
FROM artists a1
JOIN artists a2
  ON a2.country = a1.country
 AND a2.artist_id > a1.artist_id
ORDER BY a1.country, a1.artist_name
""", limit=25)

22 pairs. The aliases are not optional any more: with the same table on
both sides, `a1` and `a2` are the only way to say which copy you mean.

**Why `>` and not `<>`?** Count it with each condition:

| Condition | Rows | What is wrong with it |
|---|---|---|
| country only | 84 | Every artist is paired with itself (40 rows), and every pair appears twice |
| country and `<>` | 44 | No self-pairs, but A with B and B with A are both there |
| country and `>` | 22 | Each pair exactly once |

22 is checkable by hand: the US and the UK have 4 artists each, which is 6
pairs each. The Netherlands has 3 artists, so 3 pairs. Seven countries have
2 artists, one pair each. 6 + 6 + 3 + 7 = 22. Countries with one artist
give no pairs at all, because it is an inner join.

In [ ]:
run("""
SELECT a1.country, a1.genre, a1.artist_name AS artist, a2.artist_name AS also
FROM artists a1
JOIN artists a2
  ON a2.country = a1.country
 AND a2.genre = a1.genre
 AND a2.artist_id > a1.artist_id
""")

Three pairs share a country and a genre: two Electronic pairs, in the
Netherlands and Germany, and two American rock bands.

## Stretch 3: 2024 against 2025

Three CTEs, each building on the ones above it.

In [ ]:
run("""
WITH minutes_2024 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at < '2025-01-01'
    GROUP BY artist_id
),
minutes_2025 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at >= '2025-01-01'
    GROUP BY artist_id
),
compared AS (
    SELECT a.artist_name,
           COALESCE(y1.minutes, 0.0) AS min_2024,
           COALESCE(y2.minutes, 0.0) AS min_2025
    FROM artists a
    LEFT JOIN minutes_2024 y1 USING (artist_id)
    LEFT JOIN minutes_2025 y2 USING (artist_id)
)
SELECT artist_name, min_2024, min_2025,
       ROUND(min_2025 - min_2024, 1) AS change
FROM compared
ORDER BY change DESC
""", limit=40)

Mireille Basset is the biggest riser (+71.7 minutes) and Glass Tram the
biggest faller (-84.0). That drop cost Glass Tram the top spot: it was my
most-listened artist in 2024, and DJ Kompas overtook it in 2025. The total fell from 4596.6 minutes in 2024 to 4383.8 in 2025.

Why each piece is there:

* **Aggregate first.** Each year is reduced to one row per artist before
  anything is joined, so nothing can fan out.
* **Start from `artists` and left join both years.** If you join
  `minutes_2024` to `minutes_2025` with an inner join you get 35 rows, and
  the three artists who dropped to zero vanish. They are exactly the ones
  the question is about.
* **`COALESCE(..., 0.0)`.** Neon Harbour has no 2025 row, so its 2025
  minutes are `NULL`, and `NULL - 23.1` is `NULL`, not `-23.1`. Without
  `COALESCE` its change would be blank and it would sink to the bottom of
  the list, looking like missing data instead of a real drop.
* **`ROUND` on the change as well.** Subtracting two decimals can give
  something like `2.4000000000000004`. That is how computers store
  decimals, not a mistake in the query.

**Harder:** the artists I stopped listening to. `compared` is already
there, so the answer is one `WHERE` on it.

In [ ]:
run("""
WITH minutes_2024 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at < '2025-01-01'
    GROUP BY artist_id
),
minutes_2025 AS (
    SELECT artist_id, ROUND(SUM(minutes_played), 1) AS minutes
    FROM plays
    WHERE played_at >= '2025-01-01'
    GROUP BY artist_id
),
compared AS (
    SELECT a.artist_name,
           COALESCE(y1.minutes, 0.0) AS min_2024,
           COALESCE(y2.minutes, 0.0) AS min_2025
    FROM artists a
    LEFT JOIN minutes_2024 y1 USING (artist_id)
    LEFT JOIN minutes_2025 y2 USING (artist_id)
)
SELECT artist_name, min_2024
FROM compared
WHERE min_2024 > 0 AND min_2025 = 0
ORDER BY min_2024 DESC
""")

Neon Harbour, Amara Diallo and Odessa Brass: the three from stretch 1 that
were not "never played". This is what CTEs buy you. Once a step has a
name, the next question is a small change at the bottom, not a new query.

## Stretch 4: spot the bug

The broken query, run:

In [ ]:
run("""
SELECT a.genre,
       COUNT(w.award_id) AS awards,
       COUNT(p.play_id)  AS plays
FROM artists a
LEFT JOIN awards w USING (artist_id)
LEFT JOIN plays  p USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

Folk claims 1,095 awards. There are only 33 awards in the whole table.

Both columns fan out, and they multiply each other. Every award meets
every play of the same artist, so Fjord & Flint alone contributes
3 × 237 = 711 rows. In that block, `COUNT(w.award_id)` counts 711 awards
and `COUNT(p.play_id)` counts 711 plays.

The nasty part is that the damage is uneven. Afrobeat and Pop have the
right play counts (180 and 302), because their artists have at most one
award each. Jazz is out by exactly 1. If you had spot-checked one of
those genres, you would have concluded the query was fine.

In [ ]:
run("""
SELECT a.genre,
       COUNT(DISTINCT w.award_id) AS awards,
       COUNT(DISTINCT p.play_id)  AS plays
FROM artists a
LEFT JOIN awards w USING (artist_id)
LEFT JOIN plays  p USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

Electronic 10 and Folk 9 awards, which add up with the rest to 33. The
plays add up to 2,183, and they match the per-genre counts from the live
exercise.

**Why does `DISTINCT` work here, when `SUM(DISTINCT)` did not?** Because
`award_id` and `play_id` are keys: every real award has a different id.
The only way to see the same id twice is to see the same row twice, so
removing repeats removes exactly the copies the join made and nothing
else.

`minutes_played` is a measurement, not a key. Two different plays can last
4.65 minutes each, so `SUM(DISTINCT minutes_played)` throws away real
plays as well as copies.

**Rule:** `DISTINCT` on a key is a safe repair for a count. `DISTINCT` on
a value never is.

**Harder:** to add minutes as well, `DISTINCT` cannot help, so it is back
to aggregating first. Same recipe as the live exercise, at genre level:

In [ ]:
run("""
WITH plays_per_artist AS (
    SELECT artist_id,
           COUNT(*)            AS plays,
           SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
awards_per_artist AS (
    SELECT artist_id, COUNT(*) AS awards
    FROM awards
    GROUP BY artist_id
)
SELECT a.genre,
       COALESCE(SUM(aw.awards), 0)           AS awards,
       COALESCE(SUM(pp.plays), 0)            AS plays,
       ROUND(COALESCE(SUM(pp.minutes), 0), 1) AS minutes
FROM artists a
LEFT JOIN plays_per_artist  pp USING (artist_id)
LEFT JOIN awards_per_artist aw USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

Same awards, same plays as the `DISTINCT` version, so two different
methods agree. That agreement is the check.

## Stretch 5: the top artist in every genre

In [ ]:
run("""
WITH per_artist AS (
    SELECT artist_id, SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
ranked AS (
    SELECT a.genre,
           a.artist_name,
           ROUND(s.minutes, 1) AS minutes,
           RANK() OVER (PARTITION BY a.genre ORDER BY s.minutes DESC) AS position
    FROM per_artist s
    JOIN artists a USING (artist_id)
)
SELECT genre, artist_name, minutes
FROM ranked
WHERE position = 1
ORDER BY minutes DESC
""")

Eight genres, eight winners. Glass Tram (1175.1) and Fjord & Flint
(846.2) are the same numbers as in notebook 01, which is a free check.

**Why can the `WHERE position = 1` not go in the same `SELECT` as the
`RANK()`?** SQLite refuses with `misuse of aliased window function
position`. Think back to the execution order from session 6: `WHERE`
runs before `SELECT`. When `WHERE` runs, `position` does not exist yet.
Putting the ranking in a CTE first means it is an ordinary column by the
time the outer `WHERE` sees it.

**Why not `GROUP BY genre` with `MAX(minutes)`?** That tells you the
winning number, not who won. And it cannot do the next part at all.

**Harder:** the top two per genre is one character.

In [ ]:
run("""
WITH per_artist AS (
    SELECT artist_id, SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
ranked AS (
    SELECT a.genre,
           a.artist_name,
           ROUND(s.minutes, 1) AS minutes,
           RANK() OVER (PARTITION BY a.genre ORDER BY s.minutes DESC) AS position
    FROM per_artist s
    JOIN artists a USING (artist_id)
)
SELECT genre, position, artist_name, minutes
FROM ranked
WHERE position <= 2
ORDER BY genre, position
""", limit=16)

Electronic is the close race: Glass Tram 1175.1, DJ Kompas 1169.2, less
than six minutes apart over two years. Pop is the opposite: Sara Lindqvist
768.6, and second place only 85.9.

If two artists tied, `RANK()` would give them both 1 and skip 2.
`ROW_NUMBER()` would number them 1 and 2 anyway, picking one at random.
Which one you want depends on the question, again.

## Stretch 6: when was I halfway?

In [ ]:
run("""
WITH monthly AS (
    SELECT strftime('%Y-%m', played_at) AS month,
           SUM(minutes_played)          AS minutes
    FROM plays
    GROUP BY month
)
SELECT month,
       ROUND(minutes, 1)                            AS minutes,
       ROUND(SUM(minutes) OVER (ORDER BY month), 1) AS running_total
FROM monthly
ORDER BY month
""", limit=24)

24 months, and the last running total is 8980.4, the same as the true
total from notebook 01. Again: a free check, so take it.

**Why round at the end and not in `monthly`?** Try it. If `monthly`
rounds each month first, the last running total comes out as 8980.7. 24
small rounding errors added up into a visible one. Keep full precision
while calculating and round only for display.

Now the halfway month:

In [ ]:
run("""
WITH monthly AS (
    SELECT strftime('%Y-%m', played_at) AS month,
           SUM(minutes_played)          AS minutes
    FROM plays
    GROUP BY month
),
running AS (
    SELECT month, SUM(minutes) OVER (ORDER BY month) AS running_total
    FROM monthly
),
half AS (
    SELECT SUM(minutes_played) / 2 AS halfway
    FROM plays
)
SELECT r.month, ROUND(r.running_total, 1) AS running_total,
       ROUND(h.halfway, 1) AS halfway
FROM running r
CROSS JOIN half h
WHERE r.running_total >= h.halfway
ORDER BY r.month
LIMIT 1
""")

December 2024. Halfway through the listening fell at almost exactly
halfway through the calendar, which says the two years were about
equally heavy: 4596.6 minutes against 4383.8.

Things to point at in this query:

* **Three CTEs**, each one step: months, then running totals, then the
  target. The final `SELECT` reads almost like the question.
* **`CROSS JOIN half`, with no `ON`.** A cross join pairs every row with
  every row: no matching rule at all. Usually that is fan-out on purpose,
  but `half` has exactly one row, so it just adds a `halfway` column to
  every month.
* **`>=`, `ORDER BY`, `LIMIT 1`**: the first month at or past the line.

**Harder, restart every year:**

In [ ]:
run("""
WITH monthly AS (
    SELECT strftime('%Y-%m', played_at) AS month,
           SUM(minutes_played)          AS minutes
    FROM plays
    GROUP BY month
)
SELECT month,
       ROUND(minutes, 1) AS minutes,
       ROUND(SUM(minutes) OVER (PARTITION BY substr(month, 1, 4)
                                ORDER BY month), 1) AS running_in_year
FROM monthly
ORDER BY month
""", limit=24)

`PARTITION BY` the year, exactly as stretch 5 partitioned by genre. The
total restarts in January 2025, and the two December figures are the two
yearly totals: 4596.6 and 4383.8.

In [ ]:
con.close()
print("done")